In [1]:
import os
import json
import glob
import pandas as pd
import xlrd
import re
from pathlib import Path
import numpy as np
import xlsxwriter
import openpyxl
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from openpyxl.styles import Alignment, Border, Side, PatternFill

In [2]:
def format_excel(excel_path):
    
    wb = load_workbook(excel_path)
    #ws = wb.active

    for ws in wb.worksheets:
        wb.active = ws
        # filling colors
        season_colors = {
            'Kharif': 'FFFFFFDD',   # light yellow
            'Rabi': 'FFE7FFFF',      # light cyan
            'Summer': 'FFE7FFFF'      # light cyan
        }
        header_row = 1
        max_col = ws.max_column
        col_season = {}
        current_season = None
    
        for col_idx in range(1, max_col + 1):
            cell_value = ws.cell(row=header_row, column=col_idx).value
            if cell_value is not None and cell_value in season_colors:
                current_season = cell_value
            # If the cell is blank, it belongs to the previous season (if any)
            if current_season is not None:
                col_season[col_idx] = current_season
    
        for col_idx, season in col_season.items():
            fill = PatternFill(start_color=season_colors[season],
                               end_color=season_colors[season],
                               fill_type='solid')
            col_letter = get_column_letter(col_idx)
            for cell in ws[col_letter]:
                cell.fill = fill
    
        # making border
        thin_border = Border(
            left=Side(style='thin'),
            right=Side(style='thin'),
            top=Side(style='thin'),
            bottom=Side(style='thin')
        )
        
        for row in ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=1, max_col=ws.max_column):
            for cell in row:
                cell.border = thin_border
    
        # rotate row 2
        rotation_alignment = Alignment(textRotation=90, horizontal='center', vertical='bottom')
        for cell in ws[2]:
            cell.alignment = rotation_alignment
        
        # Set column widths (add a little padding)
        for col_index in range(1, ws.max_column+1):
            col_letter = get_column_letter(col_index)
            if col_index == 2:
                ws.column_dimensions[col_letter].width = 20
            else:
                ws.column_dimensions[col_letter].width = 3.6
    
    wb.save(excel_path)

In [3]:
def reorder_columns(df, season_crop_order):
    if df.empty:
        return df

    current_cols = df.columns
    existing = set(current_cols)
    type_order = ['UI', 'IR']  # UI first, then IR

    new_order = []
    for season in ['Kharif', 'Rabi']:   # Keep this order for seasons
        crop_list = season_crop_order.get(season, [])
        for crop in crop_list:
            for ctype in type_order:
                if (season, crop, ctype) in existing:
                    new_order.append((season, crop, ctype))
        season_cols = [col for col in current_cols if col[0] == season]
        ordered_crops = set(crop_list)
        remaining_crops = sorted([col for col in season_cols if col[1] not in ordered_crops],
                                 key=lambda x: x[1])  # alphabetical by crop
    # Build mapping for crop order per season
    crop_rank = {}
    for season, crops in season_crop_order.items():
        crop_rank[season] = {crop: idx for idx, crop in enumerate(crops)}

    type_rank = {t: i for i, t in enumerate(type_order)}

    def sort_key(col):
        season, crop, ctype = col
        # Season: Kharif first, Rabi second (ensure order)
        season_rank = 0 if season == 'Kharif' else 1
        # Crop rank: if in custom list, use its index; else use large number + alphabet
        crop_rank_for_season = crop_rank.get(season, {})
        if crop in crop_rank_for_season:
            crop_rank_val = crop_rank_for_season[crop]
        else:
            crop_rank_val = 1000 + ord(crop[0]) if crop else 999  # fallback
        # Type rank: UI=0, IR=1, others alphabetically after that
        if ctype in type_rank:
            type_rank_val = type_rank[ctype]
        else:
            type_rank_val = 10 + ord(ctype[0]) if ctype else 999
        return (season_rank, crop_rank_val, type_rank_val)

    sorted_cols = sorted(current_cols, key=sort_key)
    return df[sorted_cols]

In [4]:
def correct_distt_name(incorrect_distt):
    with open("distt_correction.json", "r") as file:
        distt_correction = json.load(file)

    for correct, incorrect_list in distt_correction.items():
        for incorrect in incorrect_list:
            if incorrect in incorrect_distt:
                correct_distt = correct
                return correct_distt

    return incorrect_distt

In [5]:
def correct_crop(incorrect_crop):
    with open("crop_correction.json", "r") as file:
        crop_correction = json.load(file)

    for correct, incorrect_list in crop_correction.items():
        for incorrect in incorrect_list:
            if incorrect.lower() in incorrect_crop.lower():
                if '(UI)' in incorrect_crop:
                    crop = correct + '_UI'
                elif '(I)' in incorrect_crop or '(IR)' in incorrect_crop:
                    crop = correct + '_IR'
                else:
                    crop = correct
                return crop
    
    return incorrect_crop

In [6]:
def get_season(crop):
    with open("crop_seq.json", "r") as file:
        crop_seq = json.load(file)

    for season, crop_dict in crop_seq.items():
        for cr in crop_dict.keys():
            if crop.lower() in cr.lower():
                return season
    return None

In [7]:
def get_first_row(worksheet, row_text_list):
    for text in row_text_list:
        text = text.lower()
        for row in worksheet.iter_rows():
            for cell in row:
                if cell.value and text in str(cell.value).lower():
                    return cell.row
    return -1

In [8]:
def get_first_col(worksheet, col_text_list):
    for text in col_text_list:
        text = text.lower()
        for row in worksheet.iter_rows():
            for cell in row:
                if cell.value and text in str(cell.value).lower():
                    return cell.column
    return -1

In [9]:
def get_SL_to_ML(directory, state):
    records = []

    #df to use
    df_state = pd.read_excel("ML_Template.xlsx", sheet_name="State")
    df_seasons = pd.read_excel("ML_Template.xlsx", sheet_name="Seasons")
    df_samples = pd.read_excel("ML_Template.xlsx", sheet_name="Samples")
    df_districts = pd.read_excel("ML_Template.xlsx", sheet_name="Districts")
    df_crops = pd.read_excel("ML_Template.xlsx", sheet_name="Crops")

    #df to create
    df_Vill = pd.DataFrame(columns=['STATENAME','SEASONNAME','SAMPLENAME','DISTRICTNAME',
                                    'CROPNAME','TALUKA','CIRCLE','VILLAGE'])
    
    df_ML = pd.DataFrame(columns=['YEAR','SEASONCODE','SEASONNAME','HSEASONNAME',
                                  'SAMPLE','SAMPLENAME','HSAMPLENAME','STATE',
                                  'STATENAME','SHORTSTATE','HSTATENAME','ROCODE',
                                  'RONAME','HRONAME','SROCODE','SRONAME','HSRONAME',
                                  'DISTRICT','DISTRICTNAME','HDISTRICTNAME',
                                  'SELORDER','EXPT','CROPCODE','CROPNAME',
                                  'HCROPNAME','STATUS','EXPTID',])
    
    # Get all .xlsx files in the directory
    excel_files = glob.glob(os.path.join(directory, "*.xlsx"))

    #for each excel workbook
    for file_path in excel_files:
        file_name = os.path.basename(file_path)
        incorrect_distt = Path(file_name).stem.title()

        distt = correct_distt_name(incorrect_distt)     ############################## usable
        # Determine file type and use appropriate library
        if file_path.endswith('.xlsx'):
            try:
                wb = load_workbook(file_path, data_only=True)
                #for each sheet in excel workbook
                for sheet_name in wb.sheetnames:
                    if "cent" in sheet_name.lower() or "sta" in sheet_name.lower():
                        st = 1 if "cent" in sheet_name.lower() else 2     ############################## usable
                        sample = df_samples[df_samples['SAMPLE']==st]['SAMPLENAME'].iloc[0]
                        
                        ws = wb[sheet_name]
                        
                        # setting which cells to scan
                        row_text_list = ['taluka', 'circle', 'vill', 'exp', 'os']
                        start_row = 0
                        title_row = 0
                        while start_row <= 0:
                            title_row = get_first_row(ws, row_text_list)
                            crop_row = title_row-1
                            start_row = title_row + 1
                            
                        col_text_list = ['vill']
                        start_col = 0
                        while start_col <= 0:
                            start_col = get_first_col(ws, col_text_list) + 1
                            taluka_col = start_col - 3
                            circle_col = start_col - 2
                            village_col = start_col - 1
                        
                        end_row = ws.max_row + 1
                        end_col = ws.max_column + 1

                        if start_row > 0 and start_col > 0:
                            for col_idx in range(start_col, end_col):
                                # get crop name and type
                                
                                incorrect_crop = ws.cell(row=crop_row, column=col_idx).value     ############################## usable
                                if isinstance(incorrect_crop, str) and incorrect_crop is not None:
                                    incorrect_crop = re.sub(r'\s+', '', incorrect_crop)
                                else:
                                    incorrect_crop = ws.cell(row=crop_row, column=col_idx-1).value
                                    if isinstance(incorrect_crop, str) and incorrect_crop is not None:
                                        incorrect_crop = re.sub(r'\s+', '', incorrect_crop)
                                    else:
                                        incorrect_crop = None
    
                                # set season
                                if incorrect_crop is not None:
                                    crop = correct_crop(incorrect_crop)
                                    crop_new = crop
                                    season = get_season(crop)
                                    
                                    
                                    plan = 0
                                    for row_idx in range(start_row, end_row):
                                        cell_val = ws.cell(row=row_idx, column=col_idx).value
                                        if isinstance(cell_val, str) and cell_val is not None:
                                            cell_val = cell_val.strip()
                                            
                                        cell_header = ws.cell(row=title_row, column=col_idx).value
                                        if isinstance(cell_header, str) and cell_header is not None:
                                            cell_header = cell_header.strip()
                                        
                                        if cell_val is not None:
                                            if taluka_col>0 and circle_col>0 and village_col>0:
                                                taluka = ws.cell(row=row_idx, column=taluka_col).value     ############################## usable
                                                circle = ws.cell(row=row_idx, column=circle_col).value     ############################## usable
                                                village = ws.cell(row=row_idx, column=village_col).value     ############################## usable

                                                if taluka is not None and circle is not None and village is not None:
                                                    if 'exp' in str(cell_header).lower():
                                                        plan += cell_val     ############################## usable
                                                    if 'os' in str(cell_header).lower():
                                                        if 'A' in str(cell_val):
                                                            crop_new = crop + '_A'
                                                        elif 'B' in str(cell_val):
                                                            crop_new = crop + '_B'

                                                        m_digit  = re.search(r"\d+", str(cell_val))
                                                        if m_digit is not None:
                                                            selorder = m_digit.group()

                                                            if len(selorder) == 1:
                                                                selorder = '0' + selorder
    
    
                                                            if state is not None:
                                                                state = state.upper()
                                                            if season is not None:
                                                                season = season.upper()
                                                            if sample is not None:
                                                                sample = sample.upper()
                                                            if distt is not None:
                                                                distt = distt.upper()
                                                            if crop_new is not None:
                                                                crop_new = crop_new.upper()
                                                            if str(taluka) is not None:
                                                                taluka = taluka.upper()
                                                            if str(circle) is not None:
                                                                circle = circle.upper()
                                                            if str(village) is not None:
                                                                village = village.upper()
                                                            
                                                            df_village_row = {
                                                                'STATENAME': state,
                                                                'SEASONNAME': season,
                                                                'SAMPLENAME': sample,
                                                                'DISTRICTNAME': distt,
                                                                'CROPNAME': crop_new,
                                                                'TALUKA': taluka,
                                                                'CIRCLE': circle,
                                                                'VILLAGE': village
                                                            }
                                                            df_Vill = pd.concat([df_Vill, pd.DataFrame([df_village_row])], ignore_index=True)
                                                            
                                                            df_ML_row_1 = {
                                                                'STATENAME': state,
                                                                'SEASONNAME': season,
                                                                'SAMPLENAME': sample,
                                                                'DISTRICTNAME': distt,
                                                                'CROPNAME': crop_new,
                                                                'SELORDER': selorder,
                                                                'EXPT': str(1)
                                                            }
                                                            
                                                            df_ML_row_1 = {
                                                                'STATENAME': state,
                                                                'SEASONNAME': season,
                                                                'SAMPLENAME': sample,
                                                                'DISTRICTNAME': distt,
                                                                'CROPNAME': crop_new,
                                                                'SELORDER': selorder,
                                                                'EXPT': str(2)
                                                            }
                                                            
                                                            df_ML = pd.concat([df_ML, pd.DataFrame([df_ML_row_1])], ignore_index=True)
                                                            df_ML = pd.concat([df_ML, pd.DataFrame([df_ML_row_1])], ignore_index=True)

            except Exception as e:
                print(f"Error reading {file_name}, {season}, {sample}, {distt}, {crop_new} (openpyxl): {e}")
                raise e
    pd.merge(df_ML,df_state,how='left',on='STATENAME')
    pd.merge(df_ML,df_seasons,how='left',on='SEASONNAME')
    pd.merge(df_ML,df_samples,how='left',on='SAMPLENAME')
    pd.merge(df_ML,df_districts,how='left',on='DISTRICTNAME')
    pd.merge(df_ML,df_crops,how='left',on='CROPNAME')
    return df_Vill, df_ML

In [10]:
if __name__ == "__main__":
    state = 'GUJARAT'
    curr_dir = Path.cwd()
    directory = curr_dir / "SL 2.0"
    excel_path = curr_dir / "SL_to_ML.xlsx"

    df_Vill, df_ML = get_SL_to_ML(directory, state)

    with pd.ExcelWriter(excel_path, engine='xlsxwriter') as writer:
        df_Vill.to_excel(writer, sheet_name="Villages", index=False)
        df_ML.to_excel(writer, sheet_name="ML", index=False)